# Internet-shopping optimization problem

Let us take a deep dive into the so-called "internet-shopping optimization problem", also known as "supplier selection and order allocation". This is a very common problem that occurs basically whenever a large number of items needs to be ordered from a large number of suppliers. As it involves *integer* quantities, it is usually formulated as a *mixed-integer linear problem* (MILP). Unfortunately, MILPs are known to be [NP-hard](https://en.wikipedia.org/wiki/NP-hardness) - which more or less says that they can be *very* hard to solve. 

## The problem as a MILP

### Variables

The variables of our optimization problems are given in the following table:

| symbol | values | meaning | cost it carries |
|---|---|---|---|
| $z_v$ | 0 or 1 | variant $v$ is used no/yes | $C^{\text{choice}}_v$ |
| $x_o$ | 0 or 1 | offer $o$ is used no/yes | $C^{\text{use}}_o$ |
| $n_o$ | $0, 1, 2, \dots$ | the number of packs bought from offer $o$ | $C^{\text{pack}}_o$ per pack |
| $u_k$ | 0 or 1 | supplier $k$ is used no/yes (that is, any of its offers) | $C^{\text{fee}}_k$|
| $f_k$ | 0 or 1 | supplier $k$ ships for free no/yes | $-C^{\text{ship}}_k$ (a saving) |

### Objective function

The problem has four "levels": **items** to buy, **variants** of each item, **offers** for each variant, and finally the **suppliers** offering the offers. The cost function has two main terms:

$$
\min_{\boldsymbol{z}, \boldsymbol{x}, \boldsymbol{n}, \boldsymbol{u}, \boldsymbol{k}}\; \sum_{i \in \mathcal{I}}\sum_{v \in \mathcal{V}_i}\Bigl(C^{\text{choice}}_v\, z_v \;+\; \sum_{o \in \mathcal{O}_v}\bigl(C^{\text{pack}}_o\, n_o + C^{\text{use}}_o\, x_o\bigr)\Bigr) \;+\; \sum_{k \in \mathcal{K}}\bigl(C^{\text{fee}}_k\, u_k - C^{\text{ship}}_k\, f_k\bigr).
$$

This means that for every item $i$ and every variant $v$ we pay $C^{\text{choice}}_v$ if the variant is chosen, and also for every
used offer $o$ of that variant we pay $C^{\text{pack}}_o$ per pack bought as well as the
fixed cost $C^{\text{use}}_o$. The second sum is over the
suppliers: the charge $C^{\text{fee}}_k$ if the supplier is used, less the
shipping $C^{\text{ship}}_k$ if shipping is free.

### Constraints

This problem has many constraints that restrict the descision variables meaningfully. For $\boldsymbol{z}$, $\boldsymbol{x}$ and $\boldsymbol{n}$, we have the following five:

| row | | for every | known as | in words |
|---|---|---|---|---|
| (1) | $N^{\min}_o\, x_o \le n_o \le N^{\max}_o\, x_o$ | offer $o \in \mathcal{O}$ | semi-integer variables, big-M rows | Packs come only from used offers, between the least and the most the offer allows. |
| (2) | $\sum_{o \in \mathcal{O}_v} \bigl(Q^{\text{pack}}_o n_o - Q^{\text{lost}}_o x_o\bigr) \ge Q^{\text{need}}_v\, z_v$ | variant $v \in \mathcal{V}$ | demand-satisfaction (covering) row, switched on by $z_v$ | The chosen variant must be covered. Each pack yield $Q^{\text{pack}}_o$ units and  loses $Q^{\text{lost}}_o$. For example, if $Q^{\text{need}}_v = 4$  and  $Q^{\text{lost}}_o = 2$, the row reads $n_o - 2x_o \ge 4$, so 6 packs are needed. A second offer used would lose its $Q^{\text{lost}}$ as well. For a variant not chosen, the right-hand side is 0, so buying nothing is allowed. |
| (3) | $\sum_{v \in \mathcal{V}_i} z_v = 1$ | item $i \in \mathcal{I}$ | exactly-one row (assignment, set partitioning) | Each item has exactly one variant chosen. |
| (4) | $\sum_{o \in \mathcal{O}_e} x_o \le 1$ | exclusive set $e \in \mathcal{E}$ | at-most-one row (set packing) | At most one offer of an exclusive set is used. For example, a product's prices for 1–9 packs and 10–99 packs cannot be mixed. |
| (5) | $\sum_{o \in \mathcal{O}_r} Q^{\text{pack}}_o\, n_o \le Q^{\text{stock}}_r$ | stock set $r \in \mathcal{R}$ | capacity row (integer knapsack) | Do not order more units than the supplier has in stock. |

For the supplier variables, we have two more:

| row | | for every | known as | in words |
|---|---|---|---|---|
| (6) | $\sum_{o \in \mathcal{O}_k} x_o \le \lvert \mathcal{O}_k \rvert\, u_k$ | supplier $k \in \mathcal{K}$ | linking row for the fixed charge $C^{\text{fee}}_k$, in summed big-M form | If any offer of supplier $k$ is used, $u_k$ must be 1, and the charge $C^{\text{fee}}_k$ is paid. Here $M = \lvert \mathcal{O}_k \rvert$, the number of its offers, the most the sum can be, so the row is harmless when $u_k = 1$. |
| (7) | $\sum_{o \in \mathcal{O}_k} \bigl(S^{\text{pack}}_o n_o + S^{\text{use}}_o x_o\bigr) \ge S^{\text{free}}_k\, f_k$ | supplier $k \in \mathcal{K}$ | threshold row, the big-M form of "if $f_k = 1$, the spend is at least $S^{\text{free}}_k$" | Shipping can be free, $f_k = 1$, only if the spend with supplier $k$ reaches $S^{\text{free}}_k$. With $f_k = 0$ the row holds trivially. |

## CP-SAT

In [ ]:
# not-rendered
INSTANCE = "p02123"
DATA = "../data"

In [ ]:
# requires %pip install "quicopt[mathopt,pyomo]"
from ortools.linear_solver.python import model_builder as mb
import gzip

# define the model and load the instance data (21538 variables, 35578 constraint rows)
model = mb.Model()
model.import_from_mps_string(gzip.open(f"{DATA}/{INSTANCE}.mps.gz", "rt").read())

solver = mb.Solver("CP-SAT")
solver.set_time_limit_in_seconds(60)
WORKERS = 8 # number of CP-SAT "workers" (roughly equal to number of CPUs)
solver.set_solver_specific_parameters(f"num_workers:{WORKERS}")
status = solver.solve(model)

print(f"status    {status.name}")
print(f"cost      {solver.objective_value:,.0f}")
print(f"time      {solver.wall_time:.1f} s on {WORKERS} workers")

In [ ]:
# not-rendered
milp = model.export_to_proto() # the model as plain data: variables and rows
print(f"{len(milp.variable)} variables, {len(milp.constraint)} rows")

## Quicopt

<!-- not-rendered -->
### Sets and parameters

A Pyomo variable is indexed by one key, so each variant is indexed by a name that is
unique across the whole instance, `item/variant`, and each offer by `item/variant/offer`.

| code | symbol | meaning |
|---|---|---|
| `I`, `V`, `O`, `K` | $\mathcal{I}$, $\mathcal{V}$, $\mathcal{O}$, $\mathcal{K}$ | all items, variants, offers, suppliers |
| `V_i[i]`, `O_v[v]`, `O_k[k]` | $\mathcal{V}_i$, $\mathcal{O}_v$, $\mathcal{O}_k$ | the variants of item `i`, the offers for variant `v`, the offers of supplier `k` |
| `E`, `O_e[e]` | $\mathcal{E}$, $\mathcal{O}_e$ | the exclusive sets, numbered from 0; the offers of exclusive set `e`, of which at most one is used |
| `R`, `O_r[r]`, `Q_stock[r]` | $\mathcal{R}$, $\mathcal{O}_r$, $Q^{\text{stock}}_r$ | the stock sets, numbered from 0; the offers of stock set `r`; the units in stock for them |

In [ ]:
# not-rendered
import json

# loading all the data and defining the sets and parameters
data = json.load(open(f"{DATA}/{INSTANCE}.json"))

variant_by_name = {f"{item['id']}/{variant['id']}": variant for item in data["items"] for variant in item["variants"]}
offer_by_name = {f"{v}/{offer['id']}": offer for v, variant in variant_by_name.items() for offer in variant["offers"]}
stock_sets = [(v, stock) for v, variant in variant_by_name.items() for stock in variant["stock"]]

V_i = {item["id"]: [f"{item['id']}/{variant['id']}" for variant in item["variants"]] for item in data["items"]}
O_v = {v: [f"{v}/{offer['id']}" for offer in variant["offers"]] for v, variant in variant_by_name.items()}
O_k = {supplier["id"]: [o for o, offer in offer_by_name.items() if offer["supplier"] == supplier["id"]] for supplier in data["suppliers"]}
O_e = dict(enumerate([f"{v}/{o}" for o in exclusive] for v, variant in variant_by_name.items() for exclusive in variant["exclusive"]))
O_r = {r: [f"{v}/{o}" for o in stock["offers"]] for r, (v, stock) in enumerate(stock_sets)}
Q_stock = {r: stock["limit"] for r, (v, stock) in enumerate(stock_sets)}

V = list(variant_by_name)
I = [item["id"] for item in data["items"]]
E = list(O_e)
R = list(O_r)
O = list(offer_by_name)
K = [supplier["id"] for supplier in data["suppliers"]]

C_choice = {v: variant["cost"] for v, variant in variant_by_name.items()}
Q_need   = {v: variant["required_units"] for v, variant in variant_by_name.items()}
N_min    = {o: offer["min_packs"] for o, offer in offer_by_name.items()}
N_max    = {o: offer["max_packs"] for o, offer in offer_by_name.items()}
C_pack   = {o: offer["cost_per_pack"] for o, offer in offer_by_name.items()}
C_use    = {o: offer["fixed_cost"] for o, offer in offer_by_name.items()}
Q_pack   = {o: offer["units_per_pack"] for o, offer in offer_by_name.items()}
Q_lost   = {o: offer["unusable_units"] for o, offer in offer_by_name.items()}
S_pack   = {o: offer["spend_per_pack"] for o, offer in offer_by_name.items()}
S_use    = {o: offer["spend_fixed"] for o, offer in offer_by_name.items()}
C_fee    = {supplier["id"]: supplier["activation_fee"] for supplier in data["suppliers"]}
C_ship   = {supplier["id"]: supplier["free_shipping_rebate"] for supplier in data["suppliers"]}
S_free   = {supplier["id"]: supplier["free_shipping_threshold"] for supplier in data["suppliers"]}

In [ ]:
import pyomo.environ as pyo
from quicopt import Client
from quicopt.structured import active, block, semiinteger

model = pyo.ConcreteModel()
model.z = pyo.Var(V, within=pyo.Binary)

# Integer numbers of packs ordered, constraint (1) via bounds and semiinteger
model.n = pyo.Var(O, within=pyo.NonNegativeIntegers, bounds=lambda model, o: (N_min[o], N_max[o]))
semiinteger(model.n)

n, z = model.n, model.z
x = {o: active(n[o]) for o in O}
for i in I:
  block(model, f"item_{i}", [n[o] for v in V_i[i] for o in O_v[v]] + [z[v] for v in V_i[i]])

In [ ]:
# constraint (2)
model.covered = pyo.Constraint(V, rule=lambda model, v: sum(Q_pack[o] * n[o] - Q_lost[o] * x[o] for o in O_v[v]) >= Q_need[v] * z[v])

# constraint (3)
model.one_variant = pyo.Constraint(I, rule=lambda model, i: sum(z[v] for v in V_i[i]) == 1)

# constraint (4)
model.exclusive = pyo.Constraint(E, rule=lambda model, e: sum(x[o] for o in O_e[e]) <= 1)

# constraint (5)
model.stock = pyo.Constraint(R, rule=lambda model, r: sum(Q_pack[o] * n[o] for o in O_r[r]) <= Q_stock[r])

pyo_obj_expr = (sum(C_choice[v] * z[v] for v in V)
              + sum(C_pack[o] * n[o] + C_use[o] * x[o] for o in O)
              + sum(pyo.Expr_if(IF=sum(x[o] for o in O_k[k]) >= 1,
                                THEN=C_fee[k],
                                ELSE=0) for k in K)  # constraint (6)
              + sum(pyo.Expr_if(IF=sum(S_pack[o] * n[o] + S_use[o] * x[o] for o in O_k[k]) >= S_free[k],
                                THEN=-C_ship[k],
                                ELSE=0) for k in K)) # constraint (7)

model.cost = pyo.Objective(sense=pyo.minimize, expr=pyo_obj_expr)

result = Client().solve(model, time_limit=2.0, gzip=True)
print(result.display)